In [1]:
#加载环境变量
from dotenv import load_dotenv
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

In [2]:
# 1.使用tool装饰器定义工具
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """
    Get the weather in a given location.
    Args:
        location: city name or coordinates
    """
    return f"Current weather in {location} is sunny"

In [5]:
from langchain.agents import create_agent
from langchain_core.messages import AIMessage,SystemMessage,HumanMessage

agent = create_agent(model="deepseek-chat",
                     tools= [get_weather]
                     )

#调用agent
for message in agent.invoke({"messages":[HumanMessage("常州今天天气怎么样")]})['messages']:
    message.pretty_print()

================================ Human Message =================================

常州今天天气怎么样
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_N6Ve1hRDwgyBlTIqKvuY9436)
 Call ID: call_00_N6Ve1hRDwgyBlTIqKvuY9436
  Args:
    location: 常州
================================= Tool Message =================================
Name: get_weather

Current weather in 常州 is sunny
================================== Ai Message ==================================

常州今天天气晴朗（sunny）。需要我帮你查更详细的温度、湿度或未来几天预报吗？


In [ ]:
# 定义工具
from langchain.tools import tool
#直接依据函数默认传值
@tool
def square_root(x: float) -> float:
    """计算指定数字的平方根"""
    return x ** 0.5

In [ ]:
#- 2通过装饰器定义工具名称
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [14]:
#- 通过装饰器定义工具名称
@tool("square_root", description="Calculate the square root of a number")
def tool1(x:float) -> float:
    #计算平方根
    return x**0.5

In [27]:
#3如果要覆盖工具的入参信息则会复杂很多，我们要借助于Pydantic或JSON约束
from pydantic import BaseModel,Field
from typing import Literal

#例如一个查天气的tool

class WeatherInput(BaseModel):
    """查询天气的输入参数"""
    location: str=Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
            default='celsius',
            description="Temperature unit preference"

    )
    include_forecast: bool = Field(
        default=False,
        description="Includes 5-day forecast"
    )


In [28]:
#定义查天气的tool

@tool(args_schema = WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

In [29]:
# 调用数学工具
tool1.invoke({"x": 467})

# 调用查询天气工具
get_weather.invoke({"location": "杭州", "include_forecast": True})

'Current weather in 杭州: 22 degrees C\nNext 5 days: Sunny'

In [32]:
#测试
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage,SystemMessage,HumanMessage

agent = create_agent(model="deepseek-chat",
                     tools=[get_weather,square_root],
                     system_prompt="你是一个智能助手帮助用户解决问题"
                     )
for token, metadata in agent.stream({"messages":[HumanMessage("常州接下来几天天气怎么样？")]},stream_mode="messages"):
    print(token.content, end="", flush=True)

Current weather in 常州: 22 degrees C
Next 5 days: Sunny常州未来几天天气情况如下：

- **当前天气**：气温约 22°C
- **未来 5 天**：以晴为主

天气不错，适合出行。不过目前只返回了“晴”的概况，具体每日温度和风力等细节没有提供。如果你需要更精确的逐日预报（比如哪天最高温、是否有降温），可以告诉我，我再帮你查。